# FinSight 360: Notebook 02 - Exploratory Data Analysis & Bivariate Profiling

**Author:** FinSight 360 Analytics Team  
**Objective:** Conduct bivariate analysis exploring how loan size, pricing, debt-to-income leverage, and risk grades correlate with observed default outcomes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import load_config

config = load_config()
df = pd.read_parquet(config['paths']['processed_data'])
print(f"Loaded {len(df):,} cleaned loan records.")
df[['loan_id', 'loan_amnt', 'term', 'grade', 'int_rate', 'dti', 'annual_inc', 'is_default']].head()

## 1. Financial Indicators by Loan Outcome
Compare median and mean values for key financial attributes across Fully Paid vs Charged Off facilities.

In [ ]:
bivariate_summary = df.groupby('loan_status')[['loan_amnt', 'int_rate', 'dti', 'annual_inc']].agg(['mean', 'median']).round(2)
bivariate_summary.T

## 2. Risk Grade and Subgrade Performance
Evaluate observed default rate progression across LendingClub assigned grades A through G.

In [ ]:
grade_perf = df.groupby('grade').agg(
    loan_count=('loan_amnt', 'count'),
    total_exposure=('loan_amnt', 'sum'),
    avg_int_rate=('int_rate', 'mean'),
    default_rate=('is_default', 'mean')
).reset_index()
grade_perf['default_rate_pct'] = (grade_perf['default_rate'] * 100).round(2)
grade_perf['avg_int_rate'] = grade_perf['avg_int_rate'].round(2)
grade_perf['exposure_m'] = (grade_perf['total_exposure'] / 1e6).round(1)
grade_perf[['grade', 'loan_count', 'exposure_m', 'avg_int_rate', 'default_rate_pct']]

## 3. Loan Purpose Risk and Concentration
Examine portfolio exposure and default rates across borrower-stated loan purposes.

In [ ]:
purpose_perf = df.groupby('purpose').agg(
    loan_count=('loan_amnt', 'count'),
    total_exposure=('loan_amnt', 'sum'),
    default_rate=('is_default', 'mean')
).reset_index()
purpose_perf['exposure_share_pct'] = (purpose_perf['total_exposure'] / df['loan_amnt'].sum() * 100).round(2)
purpose_perf['default_rate_pct'] = (purpose_perf['default_rate'] * 100).round(2)
purpose_perf = purpose_perf.sort_values(by='total_exposure', ascending=False)
purpose_perf[['purpose', 'loan_count', 'exposure_share_pct', 'default_rate_pct']]

## 4. Geographic State Concentration (Top 10 States)
Examine borrower geographical exposure across the top 10 contributing states.

In [ ]:
state_perf = df.groupby('addr_state').agg(
    loan_count=('loan_amnt', 'count'),
    total_exposure=('loan_amnt', 'sum'),
    default_rate=('is_default', 'mean')
).reset_index()
state_perf['exposure_share_pct'] = (state_perf['total_exposure'] / df['loan_amnt'].sum() * 100).round(2)
state_perf['default_rate_pct'] = (state_perf['default_rate'] * 100).round(2)
state_perf = state_perf[state_perf['addr_state'] != 'UNKNOWN'].sort_values(by='total_exposure', ascending=False).head(10)
state_perf[['addr_state', 'loan_count', 'exposure_share_pct', 'default_rate_pct']]